# SLM-KDN Colab Pro A100: RAG + Training + Testing

Use this notebook after syncing your repo to `master`. It clones the GitHub repo, installs dependencies, checks the A100, rebuilds the RAG index over `rag-doc/ex3300.pdf`, runs retrieval smoke tests, then runs preprocess, training, inference, and evaluation.

Before running: in Colab, choose **Runtime -> Change runtime type -> GPU**, then pick an A100 if available.

## 1. Check GPU

In [1]:
!nvidia-smi

import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Capability:", torch.cuda.get_device_capability(0))

Mon May 11 20:58:24 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P0             45W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## 2. Clone Your Repo from Master

In [2]:
%cd /content
!rm -rf slm-kdn
!git clone --branch master https://github.com/Surya-Prasad/slm-kdn.git
%cd /content/slm-kdn
!git status --short
!git rev-parse --abbrev-ref HEAD
!git log -1 --oneline

/content
Cloning into 'slm-kdn'...
remote: Enumerating objects: 184, done.
remote: Counting objects: 100% (184/184), done.
remote: Compressing objects: 100% (130/130), done.
remote: Total 184 (delta 99), reused 109 (delta 51), pack-reused 0 (from 0)
Receiving objects: 100% (184/184), 2.77 MiB | 13.49 MiB/s, done.
Resolving deltas: 100% (99/99), done.
/content/slm-kdn
master
8ef4de5 (HEAD -> master, origin/master, origin/HEAD) Update Scores


## 3. Install Dependencies

This installs the project requirements. `pypdf` is required for `rag-doc/ex3300.pdf` ingestion.

In [3]:
!pip install -q -r requirements.txt
!pip install -q --upgrade transformers peft accelerate bitsandbytes datasets pypdf

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 45.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 338.8/338.8 kB 37.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 152.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 46.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 57.3 MB/s eta 0:00:00


## 4. Hugging Face Login

Run this if your base model requires gated access. Use a token that has access to `meta-llama/Meta-Llama-3-8B`.

In [5]:
from huggingface_hub import notebook_login
notebook_login()

## 5. Confirm EX3300 Guide Is Present

In [6]:
!find rag-doc -maxdepth 2 -type f -print
!ls -lh rag-doc/ex3300.pdf

rag-doc/ex3300.pdf
-rw-r--r-- 1 root root 3.5M May 11 20:58 rag-doc/ex3300.pdf


## 6. Build/Test RAG Retrieval Only

This does not run training. It rebuilds `results/rag_index.pkl` and prints source file, page, score, and chunk preview.

In [7]:
!python src/rag_query.py --rebuild --query "What are the front panel ports on a Juniper EX3300 switch?"
!python src/rag_query.py --query "How do I interpret ALM SYS MST LEDs on an EX3300?" --top_k 3
!python src/rag_query.py --query "What are the console port details for EX3300?" --top_k 3
!python src/rag_query.py --query "What power supply information does the EX3300 guide provide?" --top_k 3

[RAG] Building index from:
[RAG]   include rag-doc/ex3300.pdf
[RAG] query: What are the front panel ports on a Juniper EX3300 switch?
[RAG] 1. source=ex3300.pdf, page=25, score=0.8500, dense=0.3985, lexical=0.2188, preview=Front Panel of an EX3300 Switch The front panel of an EX3300 switch consists of the following components: • Network ports: • Depending on the switch model, 24 or 48 10/100/1000BASE-T Gigabit Ethernet 
[RAG] 2. source=ex3300.pdf, page=26, score=0.5819, dense=0.2784, lexical=0.1028, preview=Figure 2: Front Panel of an EX3300 Switch with 24 Gigabit Ethernet Ports g021217 0 1 2 3 ALM EX3300 SYS MST LCD panel Network ports Chassis status LEDs Enter buttonSFP+ uplink ports Menu button Rear P
[RAG] 3. source=ex3300.pdf, page=82, score=0.5784, dense=0.2632, lexical=0.1148, preview=Unpacking and Mounting the EX3300 Switch IN THIS SECTION Unpacking an EX3300 Switch | 82 Parts Inventory (Packing List) for an EX3300 Switch | 83 Register Products—Mandatory to Validate SLAs | 84 I

## 7. Preprocess NIT Dataset

This preserves the original NIT pipeline and creates `data/processed/*.jsonl` files.

In [8]:
!bash scripts/run_preprocess.sh
!find data/processed -maxdepth 1 -type f -print | sort
!head -n 1 data/processed/test.jsonl

README.md: 4.31kB [00:00, 7.68MB/s]
NIT_dataset.json: 273kB [00:00, 186MB/s]
Generating train split: 100% 1000/1000 [00:00<00:00, 114279.98 examples/s]
data/processed/clean_test.jsonl
data/processed/noisy_test.jsonl
data/processed/paraphrased_test.jsonl
data/processed/test_intent_only.jsonl
data/processed/test_intent_with_context.jsonl
data/processed/test.jsonl
data/processed/train_intent_only.jsonl
data/processed/train_intent_with_context.jsonl
data/processed/train.jsonl
data/processed/val_intent_only.jsonl
data/processed/val_intent_with_context.jsonl
data/processed/val.jsonl
{"intent": "How to display the LED status", "context": "show chassis led", "target_command": "show chassis led", "target_json": "{\"action\":\"show\",\"parameters\":{\"prefix\":\"\",\"unit\":0,\"vlan_id\":null,\"vlan_name\":\"\"},\"target\":\"\",\"target_type\":\"unknown\"}", "category": ""}


## 8. Rebuild RAG After NIT Preprocessing

After preprocessing, the RAG index includes both the processed NIT JSONL records and the EX3300 guide.

In [9]:
!python src/rag_query.py --rebuild --query "What are the front panel ports on a Juniper EX3300 switch?" --top_k 5

[RAG] Building index from:
[RAG]   include data/processed/train.jsonl
[RAG]   include rag-doc/ex3300.pdf
[RAG]   exclude data/processed/val.jsonl
[RAG]   exclude data/processed/test.jsonl
[RAG] query: What are the front panel ports on a Juniper EX3300 switch?
[RAG] 1. source=ex3300.pdf, page=25, score=0.8500, dense=0.3523, lexical=0.2311, preview=Front Panel of an EX3300 Switch The front panel of an EX3300 switch consists of the following components: • Network ports: • Depending on the switch model, 24 or 48 10/100/1000BASE-T Gigabit Ethernet 
[RAG] 2. source=ex3300.pdf, page=82, score=0.6170, dense=0.2511, lexical=0.1351, preview=Unpacking and Mounting the EX3300 Switch IN THIS SECTION Unpacking an EX3300 Switch | 82 Parts Inventory (Packing List) for an EX3300 Switch | 83 Register Products—Mandatory to Validate SLAs | 84 Inst
[RAG] 3. source=ex3300.pdf, page=26, score=0.6061, dense=0.2587, lexical=0.1162, preview=Figure 2: Front Panel of an EX3300 Switch with 24 Gigabit Ethernet Port

## 9. Train LoRA on A100

The repo config is already A100-oriented. If Colab runs out of memory, lower `training.max_seq_len` or batch settings in `config.yaml` before running this cell.

In [10]:
!bash scripts/run_train.sh

config.json: 100% 654/654 [00:00<00:00, 2.89MB/s]
tokenizer_config.json: 50.6kB [00:00, 33.1MB/s]
tokenizer.json: 9.09MB [00:01, 5.05MB/s]
special_tokens_map.json: 100% 73.0/73.0 [00:00<00:00, 463kB/s]
model.safetensors.index.json: 23.9kB [00:00, 53.6MB/s]
Fetching 4 files:  25% 1/4 [00:43<02:10, 43.55s/it]
Fetching 4 files: 100% 4/4 [00:43<00:00, 10.94s/it]
Download complete: 100% 16.1G/16.1G [00:43<00:00, 367MB/s]
Loading weights: 100% 291/291 [00:04<00:00, 65.33it/s]
generation_config.json: 100% 177/177 [00:00<00:00, 852kB/s]
Map: 100% 722/722 [00:00<00:00, 2192.30 examples/s]
{'loss': '0.3458', 'grad_norm': '0.7146', 'learning_rate': '0.000187', 'epoch': '0.2216'}
{'loss': '0.1018', 'grad_norm': '0.7223', 'learning_rate': '0.0001725', 'epoch': '0.4432'}
{'loss': '0.05985', 'grad_norm': '1.469', 'learning_rate': '0.000158', 'epoch': '0.6648'}
{'loss': '0.04032', 'grad_norm': '0.1962', 'learning_rate': '0.0001435', 'epoch': '0.8864'}
{'loss': '0.03691', 'grad_norm': '0.546', 'learnin

## 10. RAG Inference Smoke Test

Create a tiny input file with EX3300-specific questions and run inference with retrieved EX3300 context. The `--rag_debug` output should show chunks from `ex3300.pdf`.

In [11]:
!pip install --upgrade torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 112.9 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [12]:
import json, os
os.makedirs("data/processed", exist_ok=True)
rows = [
    {"intent": "What are the front panel ports on a Juniper EX3300 switch?", "context": "", "target_command": "", "category": "rag_ex3300"},
    {"intent": "How do I interpret ALM SYS MST LEDs on an EX3300?", "context": "", "target_command": "", "category": "rag_ex3300"},
    {"intent": "What are the console port details for EX3300?", "context": "", "target_command": "", "category": "rag_ex3300"},
]
with open("data/processed/rag_smoke.jsonl", "w", encoding="utf-8") as f:
    for row in rows:
        f.write(json.dumps(row) + "\n")

!python src/infer.py \
  --input_file data/processed/rag_smoke.jsonl \
  --output_file results/predictions/rag_smoke_predictions.jsonl \
  --use_rag \
  --rebuild_rag \
  --rag_debug

!cat results/predictions/rag_smoke_predictions.jsonl

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
[RAG] Building index from:
[RAG]   include data/processed/train.jsonl
[RAG]   include rag-doc/ex3300.pdf
[RAG]   exclude data/processed/val.jsonl
[RAG]   exclude data/processed/test.jsonl
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 291/291 [00:04<00:00, 59.78it/s]

[INFO] Starting BATCHED inference on 3 instances for rag_smoke.jsonl...
Batches:   0% 0/1 [00:00<?, ?it/s][RAG] query: What are the front panel ports on a Juniper EX3300 switch?
[RAG] 1. source=ex3300.pdf, page=25, score=0.8500, dense=0.3523, lexical=0.2311, preview=Front Panel of an EX3300 Switch The front panel of an EX3300 switch consists of the following components: • Network ports: • Depending on the switch model, 24 or 48 10/100/1000BASE-T Gigabit Ethernet 
[RAG] 2. source=ex3300.pdf, page=82, score=0.6170, dense=0.2511, lexical=0.1351, preview=Unpacking

## 11. Full Test Inference + Evaluation

Use this for the original NIT evaluation. Keep `--use_rag` on if you want retrieved context included during inference.

In [13]:
!mkdir -p results/predictions results/metrics
!python src/infer.py \
  --input_file data/processed/test.jsonl \
  --output_file results/predictions/rag_predictions.jsonl \
  --use_rag \
  --rag_debug

!python src/evaluate.py --pred_file results/predictions/rag_predictions.jsonl
!cat results/metrics/eval_metrics.json

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 291/291 [00:05<00:00, 58.04it/s]

[INFO] Starting BATCHED inference on 150 instances for test.jsonl...
Batches:   0% 0/5 [00:00<?, ?it/s][RAG] query: How to display the LED status
[RAG] 1. source=train.jsonl, score=1.0688, dense=0.5333, lexical=0.2370, preview=Intent: Show LED status Context: show chassis led Target command: show chassis led
[RAG] 2. source=ex3300.pdf, page=39, score=0.4987, dense=0.2454, lexical=0.1405, preview=Table 12: Status LED on the Uplink Ports in EX3300 Switches State and DescriptionLCD IndicatorLED Indicates the speed. The status indicators are: • Unlit—10/100 Mbps • Blinking—1000 Mbps • On steadily
[RAG] 3. source=ex3300.pdf, page=36, score=0.4921, dense=0.2686, lexical=0.1146, preview=Table 9: Status LED on the Management Port on an EX3300 Switch St

## 12. Save Results to Google Drive

In [14]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/slm-kdn-results
!cp -r results /content/drive/MyDrive/slm-kdn-results/results-rag-$(date +%Y%m%d-%H%M%S)
!ls -lh /content/drive/MyDrive/slm-kdn-results

Mounted at /content/drive
total 24K
drwx------ 2 root root 4.0K May 11 18:43 results-rag-20260511-184340
drwx------ 2 root root 4.0K May 11 19:17 results-rag-20260511-191756
drwx------ 2 root root 4.0K May 11 19:43 results-rag-20260511-194342
drwx------ 2 root root 4.0K May 11 20:17 results-rag-20260511-201701
drwx------ 2 root root 4.0K May 11 20:57 results-rag-20260511-205733
drwx------ 5 root root 4.0K May 11 21:11 results-rag-20260511-211105
